# Lakeflow Declarative Pipelines (formerly Delta Live Tables)

Build declarative data pipelines with automatic dependency management, data quality expectations, and built-in lineage tracking.

| Training Block | Duration | Type |
|---|---|---|
| Lakeflow Pipelines — Demo | 40 min | Demo |
| Lakeflow Pipelines — Workshop | 30 min | Hands-on |

**Prerequisites:** 01 — Medallion Architecture (SCD concepts), 02 — Batch & Stream Ingestion (Auto Loader)

## Learning Objectives

After completing this module you will be able to:

- **Explain** the declarative approach of Lakeflow pipelines vs. imperative Spark code
- **Create** `STREAMING TABLE` and `MATERIALIZED VIEW` declarations
- **Apply** data quality `EXPECT` constraints with violation actions (DROP ROW, FAIL UPDATE)
- **Implement** AUTO CDC for SCD Type 1 and Type 2 using `CREATE FLOW`
- **Build** and run a Lakeflow pipeline in the Databricks UI
- **Monitor** pipeline runs, lineage graph, and event logs

---

## Lakeflow Pipelines — Declarations

Lakeflow (formerly Delta Live Tables) enables a declarative approach to building data pipelines — you define the desired outcome rather than step-by-step logic. This section covers table types, expectations, FLOW declarations, and both SQL and PySpark syntax.

---

### What is Lakeflow?

**Lakeflow** (formerly Delta Live Tables) — declarative framework for data pipelines.

| Approach | Example | You describe |
|-----------|----------|-------------|
| **Imperative** | `df.write.mode("overwrite")...` | HOW |
| **Declarative** | `CREATE TABLE AS SELECT...` | WHAT |

Key benefits: automatic dependencies, built-in data quality, unified batch/streaming, automatic recovery, lineage & monitoring.

### Table Types in Lakeflow

| Type | Usage | Processing |
|-----|--------|-----------|
| **STREAMING TABLE** | Append-only ingestion | Incremental (new rows only) |
| **MATERIALIZED VIEW** | Aggregations, Gold layer | Full recomputation |
| **VIEW** | Intermediate logic | Not materialized |

> **Key Point:** STREAMING TABLE = incremental (new data only). MATERIALIZED VIEW = full recompute each refresh.

**STREAMING TABLE vs MATERIALIZED VIEW :**

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Data Source | Streaming (append-only) | Batch or streaming |
| Processing | Incremental (new rows only) | Full recomputation |
| Updates | Append-only | Full refresh |
| Use Case | Bronze/Silver layers | Gold aggregations |
| Supports AUTO CDC | Yes | No |
| Query Pattern | `STREAM(source)` | Regular `SELECT` |
| Idempotent | Yes (checkpoints) | Yes (full refresh) |

**Key Point:** STREAMING TABLEs process only NEW data (incremental). MATERIALIZED VIEWs recompute the FULL result on each refresh.

### Demo: STREAMING TABLE with Constraints

Constraint actions: **`DROP ROW`** (remove invalid), **`FAIL UPDATE`** (fail pipeline), or no action (log only).

**Lakeflow — SQL Declarations Reference**

| Statement | Syntax |
|-----------|--------|
| Streaming Table | `CREATE OR REFRESH STREAMING TABLE name (col TYPE, CONSTRAINT c EXPECT (expr) ON VIOLATION action)` |
| Materialized View | `CREATE OR REFRESH MATERIALIZED VIEW name AS SELECT ...` |
| Expectation: warn only | *(omit ON VIOLATION)* — all rows kept, violation logged in metrics |
| Expectation: drop rows | `ON VIOLATION DROP ROW` |
| Expectation: fail update | `ON VIOLATION FAIL UPDATE` |
| Read upstream incremental | `FROM STREAM(table_name)` |
| Read upstream full scan | `FROM table_name` |
| Pipeline parameter | `'${parameter_name}'` |


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE silver_orders
(
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_customer EXPECT (customer_id IS NOT NULL)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_quantity EXPECT (quantity > 0)
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_price EXPECT (unit_price >= 0)
    ON VIOLATION FAIL UPDATE
)
AS
SELECT
  order_id,
  customer_id,
  product_id,
  CAST(order_datetime AS TIMESTAMP) AS order_ts,
  quantity,
  unit_price,
  (quantity * unit_price) AS gross_amount
FROM STREAM(bronze_orders);

### Demo: Expectations — Quarantine Pattern

Instead of dropping or failing, route invalid records to a **quarantine table** for investigation. This is the recommended production pattern for data quality management.

```
Bronze  →  [Quality Check]  →  Valid rows   → Silver
                             →  Invalid rows → Quarantine
```

> **Key Insight:** Use `EXPECT` (warn only) on the intermediate table to **log metrics** while the `is_quarantined` flag routes rows to the correct downstream table.

In [0]:
%sql
-- Step 1: Intermediate table — tag rows, log quality metrics
CREATE TEMPORARY STREAMING TABLE orders_quality_check
(
  CONSTRAINT valid_order_id EXPECT (order_id IS NOT NULL),
  CONSTRAINT valid_quantity EXPECT (quantity > 0),
  CONSTRAINT valid_price   EXPECT (unit_price >= 0)
)
AS
SELECT
  *,
  CASE
    WHEN order_id IS NULL OR quantity <= 0 OR unit_price < 0
    THEN true
    ELSE false
  END AS is_quarantined
FROM STREAM(bronze_orders);

-- Step 2: Valid records → Silver (clean data)
CREATE OR REFRESH STREAMING TABLE silver_orders_valid
AS SELECT * FROM STREAM(orders_quality_check)
WHERE is_quarantined = false;

-- Step 3: Invalid records → Quarantine (for investigation)
CREATE OR REFRESH STREAMING TABLE quarantine_orders
AS SELECT * FROM STREAM(orders_quality_check)
WHERE is_quarantined = true;

-- Pro Tip: Set up alerts on quarantine table row count
-- to notify data engineers when invalid records arrive.

### Demo: MATERIALIZED VIEW (Gold)

In [0]:
%sql
-- Dimension - current snapshot from SCD2
CREATE OR REFRESH MATERIALIZED VIEW dim_customer
AS
SELECT
  customer_id,
  first_name,
  last_name,
  email,
  city,
  customer_segment
FROM silver_customers
WHERE __END_AT IS NULL;

-- Date Dimension
CREATE OR REFRESH MATERIALIZED VIEW dim_date
AS
SELECT DISTINCT
  CAST(date_format(order_date, 'yyyyMMdd') AS INT) AS date_key,
  order_date AS date,
  year(order_date) AS year,
  quarter(order_date) AS quarter,
  month(order_date) AS month
FROM silver_orders;

-- Fact - streaming from Silver
CREATE OR REFRESH STREAMING TABLE fact_sales
AS
SELECT
  order_id,
  customer_id,
  product_id,
  order_date_key,
  quantity,
  gross_amount,
  net_amount
FROM STREAM(silver_orders);

### What is FLOW?

**FLOW** separates table definition from data source. Key capabilities:
1. **Multiple sources → one table** (e.g. backfill + streaming)
2. **CDC** with automatic SCD via `AUTO CDC`
3. **`INSERT INTO ONCE`** — one-time backfill
4. **`INSERT INTO`** — continuous streaming

In [0]:
%sql
-- 1. We define empty target table
CREATE OR REFRESH STREAMING TABLE bronze_orders;

-- 2. We define FLOW(s) which populate it
CREATE FLOW flow_name
AS INSERT INTO target_table BY NAME
SELECT ... FROM source;

### Demo: Backfill + Streaming Pattern

**FLOW — Syntax Reference**

| Statement | Syntax |
|-----------|--------|
| Empty target table | `CREATE OR REFRESH STREAMING TABLE bronze_orders;` |
| One-time backfill | `CREATE FLOW name AS INSERT INTO ONCE target BY NAME SELECT ... FROM read_files(...)` |
| Continuous stream | `CREATE FLOW name AS INSERT INTO target BY NAME SELECT ... FROM STREAM read_files(...)` |
| CDC flow (SCD1) | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM src KEYS (key) SEQUENCE BY ts STORED AS SCD TYPE 1` |
| CDC flow (SCD2) | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM src KEYS (key) SEQUENCE BY ts STORED AS SCD TYPE 2` |


In [0]:
%sql
-- Target table
CREATE OR REFRESH STREAMING TABLE bronze_orders;

-- FLOW 1: One-time backfill
CREATE FLOW bronze_orders_backfill
AS 
INSERT INTO ONCE bronze_orders BY NAME
SELECT
  order_id,
  customer_id,
  product_id,
  order_datetime,
  'batch' AS source_system,
  _metadata.file_path AS source_file_path,
  current_timestamp() AS load_ts
FROM read_files(
  '${order_path}/orders_batch.json',
  format => 'json'
);

-- FLOW 2: Continuous streaming
CREATE FLOW bronze_orders_stream
AS 
INSERT INTO bronze_orders BY NAME
SELECT
  order_id,
  customer_id,
  'stream' AS source_system,
  _metadata.file_path AS source_file_path,
  current_timestamp() AS load_ts
FROM STREAM read_files(
  '${order_path}/stream/orders_stream_*.json',
  format => 'json'
);

### What is SCD (Slowly Changing Dimension)?

**Slowly Changing Dimension (SCD)** — strategy for handling changes in dimensional data over time.

| Type | Strategy | Result | Use Case |
|------|---------|--------|----------|
| **SCD Type 0** | Retain original | Always "Warsaw" | Reference data that never changes |
| **SCD Type 1** | Overwrite | Only "Kraków" | Error corrections, non-historical data |
| **SCD Type 2** | Track full history | Both records with timestamps | Audit, compliance, historical analysis |
| **SCD Type 3** | Add column | `current_city=Kraków`, `previous_city=Warsaw` | Single step back only |

> In Lakeflow: `STORED AS SCD TYPE 1` overwrites, `STORED AS SCD TYPE 2` tracks full history with `__START_AT` / `__END_AT`.

### Demo: AUTO CDC for SCD Type 2

AUTO CDC: compares new records by `KEYS`, detects changes, closes old record (`__END_AT`), inserts new (SCD2) or overwrites (SCD1).

**AUTO CDC — Syntax Reference**

| Element | Syntax / Value |
|---------|---------------|
| Target table | `CREATE OR REFRESH STREAMING TABLE silver_customers (customer_id STRING, ..., __START_AT TIMESTAMP, __END_AT TIMESTAMP)` |
| CDC Flow | `CREATE FLOW name AS AUTO CDC INTO target FROM STREAM source` |
| Business key | `KEYS (customer_id)` |
| Change order | `SEQUENCE BY ingestion_ts` |
| SCD Type 1 (overwrite) | `STORED AS SCD TYPE 1` |
| SCD Type 2 (history) | `STORED AS SCD TYPE 2` |
| Query current records | `WHERE __END_AT IS NULL` |


In [0]:
%sql
-- SCD2 Table with schema
CREATE OR REFRESH STREAMING TABLE silver_customers (
  customer_id        STRING,
  first_name         STRING,
  last_name          STRING,
  email              STRING,
  city               STRING,
  __START_AT         TIMESTAMP,
  __END_AT           TIMESTAMP
);

-- AUTO CDC Flow
CREATE FLOW silver_customers_scd2_flow
AS AUTO CDC INTO silver_customers
FROM STREAM bronze_customers
KEYS (customer_id)
SEQUENCE BY ingestion_ts
STORED AS SCD TYPE 2;

### PySpark Declarations

Lakeflow pipelines can also be defined in Python using the `pyspark.pipelines` decorators instead of SQL.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# STREAMING TABLE
@dp.table(
    name="bronze_customers",
    comment="Raw customers from CSV"
)
def bronze_customers():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load(spark.conf.get("customer_path"))
            .select(
                "*",
                col("_metadata.file_path").alias("source_file_path"),
                current_timestamp().alias("load_ts")
            )
    )

# MATERIALIZED VIEW
@dp.table(name="dim_customer")
def dim_customer():
    return (
        spark.read.table("silver_customers")
            .filter(col("__END_AT").isNull())
            .select("customer_id", "first_name", "last_name")
    )

### PySpark: Expectations

Decorators: **`@dp.expect`** (log only), **`@dp.expect_or_drop`** (drop record), **`@dp.expect_or_fail`** (fail pipeline).

In [0]:
@dp.table(name="silver_orders")
@dp.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dp.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
@dp.expect_or_drop("valid_quantity", "quantity > 0")
@dp.expect("valid_price", "unit_price >= 0")
def silver_orders():
    return (
        spark.readStream.table("bronze_orders")
            .select(
                "order_id",
                "customer_id",
                col("order_datetime").cast("timestamp").alias("order_ts"),
                (col("quantity") * col("unit_price")).alias("gross_amount")
            )
    )

### PySpark: Reusable Expectations & Quarantine

Use **`expect_all`** / **`expect_all_or_drop`** to apply a dictionary of rules across multiple datasets. Combine with the quarantine pattern for production-grade quality management.

| Decorator | Behavior |
|---|---|
| `@dp.expect_all(rules)` | Log all violations (warn) |
| `@dp.expect_all_or_drop(rules)` | Drop rows failing ANY rule |
| `@dp.expect_all_or_fail(rules)` | Fail pipeline on ANY violation |

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import expr

# ── Reusable rules dictionary (share across datasets) ──
quality_rules = {
    "valid_order_id": "(order_id IS NOT NULL)",
    "valid_quantity": "(quantity > 0)",
    "valid_price":    "(unit_price >= 0)",
    "valid_date":     "(order_datetime IS NOT NULL)",
}

# Build quarantine expression: NOT(rule1 AND rule2 AND ...)
quarantine_expr = "NOT({0})".format(" AND ".join(quality_rules.values()))

# ── Intermediate: tag rows + log metrics ──
@dp.table(temporary=True, partition_cols=["is_quarantined"])
@dp.expect_all(quality_rules)  # logs pass/fail counts for all rules
def orders_quarantine():
    return (
        spark.readStream.table("bronze_orders")
        .withColumn("is_quarantined", expr(quarantine_expr))
    )

# ── Valid records → downstream processing ──
@dp.view
def valid_orders():
    return spark.read.table("orders_quarantine").filter("is_quarantined=false")

# ── Invalid records → quarantine for investigation ──
@dp.view
def quarantine_orders():
    return spark.read.table("orders_quarantine").filter("is_quarantined=true")

### PySpark: AUTO CDC

Python equivalent of SQL AUTO CDC — uses `dp.create_auto_cdc_flow()` to define SCD logic programmatically.

In [0]:
from pyspark import pipelines as dp

# Define the target table
dp.create_streaming_table(
    name="silver_customers",
    schema="""
        customer_id STRING,
        first_name STRING,
        city STRING,
        __START_AT TIMESTAMP,
        __END_AT TIMESTAMP
    """
)

# Define the CDC flow
dp.create_auto_cdc_flow(
    target="silver_customers",
    source="bronze_customers",
    keys=["customer_id"],
    sequence_by="ingestion_ts",
    stored_as_scd_type=2  # or 1
)

---

### What's New in Lakeflow (2025-2026)

| Feature | Status | Description |
|---------|--------|-------------|
| **`pyspark.pipelines`** | GA | New Python module replacing `import dlt` |
| **`expect_all` dictionaries** | GA | Reusable rules across multiple datasets |
| **Quarantine pattern** | Recommended | Route invalid rows to separate table for investigation |
| **Sinks API** | GA | Write pipeline output to Kafka, Event Hubs, external Delta tables |
| **ForEachBatch** | Preview (Dec 2025) | Process streams as micro-batches, write to arbitrary targets |
| **Auto Loader File Events** | GA (Dec 2025) | Efficient file discovery without cloud notification setup |
| **Lakeflow System Tables** | Preview (Dec 2025) | Enhanced monitoring: pipeline metrics, job duration breakdown |

> **Key trend:** Lakeflow pipelines are evolving toward serverless-first compute, declarative sinks for external targets, and deeper observability via system tables.

---

## Section 2: UI Demo — Building a Lakeflow Pipeline

Step-by-step walkthrough of creating, configuring, and executing a Lakeflow pipeline in the Databricks workspace. This demo covers the full lifecycle: pipeline creation, asset configuration, compute provisioning, validation (dry run), and production execution.

> **Estimated time:** ~15 min (including 5-8 min for initial cluster provisioning)

---

### Step 1: Create Pipeline

Navigate to **Jobs & Pipelines** in the left sidebar and select **ETL Pipeline**.

<img src="../../../assets/images/training_2026/day3/fd6ad124b1764ec0a2867cb7c612c75d.webp" width="800">

Assign a unique, descriptive name to your pipeline (e.g., `<your_name>_lakeflow_demo`).

<img src="../../../assets/images/training_2026/day3/eb57b69aa1494a759d1f69fc1de84189.webp" width="800">

### Step 2: Configure Target — Catalog & Schema

Select the target **Unity Catalog** catalog and schema for pipeline output tables. In a shared training environment, create a dedicated schema with a unique name (e.g., `<your_name>_lakeflow_demo`) to avoid namespace conflicts with other participants.

<img src="../../../assets/images/training_2026/day3/bef9f61c293743f190f68d188265f635.webp" width="800">

### Step 3: Add Source Assets

Click **Add existing assets** to link the pipeline to repository-managed source code.

<img src="../../../assets/images/training_2026/day3/198cba2b98dd4bafbceb4d178597cfc6.webp" width="800">

Configure the following paths:
- **Pipeline root folder** → `materials/lakeflow/lakeflow_ny_demo`
- **Source code path** → `transformations` subdirectory within `lakeflow_ny_demo`

<img src="../../../assets/images/training_2026/day3/8b6b591b1b3843f6a39fbe400448c311.webp" width="800">

Verify that the final configuration matches the expected setup before proceeding:

<img src="../../../assets/images/training_2026/day3/8300f3f7570043ceaf1411a8be91e9a7.webp" width="800">

### Step 4: Review Pipeline Structure

The left panel displays the auto-discovered pipeline graph — all tables, views, and dependencies extracted automatically from the source code repository.

<img src="../../../assets/images/training_2026/day3/1e476ee5b1f94c649511adddc93a2bfd.webp" width="800">

Navigate to the **Settings** tab to configure compute resources.

<img src="../../../assets/images/training_2026/day3/5c2bfc315a2148268f2c9bf313bbc928.webp" width="800">

### Step 5: Configure Compute Resources

Open **Compute** → **Edit compute configuration** to adjust cluster settings.

<img src="../../../assets/images/training_2026/day3/7674419b96ca494d8c9e5c0ed9bc45fe.webp" width="800">

Set **Cluster mode** to `Fixed size` with **Workers = 1** to minimize resource consumption within training quota limits.

<img src="../../../assets/images/training_2026/day3/7980e9740c6b4ab99183520e23684aeb.webp" width="800">

Under **Advanced settings**, select worker type **D4ds_v5** — optimized for the training environment's vCPU quota constraints.

<img src="../../../assets/images/training_2026/day3/c410d04e1ddd4be99b52fbb35f124bac.webp" width="800">

> **Note:** Training environments have strict vCPU quotas. Using a minimal fixed-size cluster ensures reliable execution without exceeding resource limits.

Click **Save** and close the compute configuration dialog.

### Step 6: Validate Pipeline — Dry Run

Click **Dry Run** to validate the pipeline without processing data. This verifies all SQL/Python declarations, resolves table dependencies, and confirms resource availability.

<img src="../../../assets/images/training_2026/day3/b06e4093070d4febababc2678f560ead.webp" width="800">

> **Tip:** Initial cluster provisioning may take 5-8 minutes. This is expected for the first run — use this time for a break.

After successful validation, the pipeline DAG is generated — showing all tables, dependencies, and data flow paths from source to gold layer.

<img src="../../../assets/images/training_2026/day3/0ebbedde030b43e7bed35295000c42a9.webp" width="800">

### Step 7: Execute Pipeline — Full Refresh

With validation confirmed, click **Run Pipeline** → select **Full table refresh** to execute the complete pipeline end-to-end.

<img src="../../../assets/images/training_2026/day3/075c2f87b48d4e38ab018495c631aaa7.webp" width="800">

A successful execution shows all tables with **green status indicators** and a fully green DAG — confirming that data has been processed through the complete **Bronze → Silver → Gold** medallion architecture.

<img src="../../../assets/images/training_2026/day3/044431e4088049faa10fb2aee4e9d84e.webp" width="800">

> **Result:** The Lakeflow pipeline has been executed end-to-end — from raw source ingestion (Bronze) through validated transformations (Silver) to business-ready aggregations (Gold) — using the declarative framework with automatic dependency management and data quality enforcement.

## APPLY CHANGES INTO — Legacy DLT SQL Syntax

> **Pro Tip:** Databricks documentation references both `APPLY CHANGES INTO` (legacy DLT SQL) and `AUTO CDC` (Lakeflow/modern DLT syntax). They are functionally equivalent — `AUTO CDC` is the preferred modern form, but `APPLY CHANGES INTO` still appears in many code samples and existing pipelines.

### Syntax Comparison

| Modern (Lakeflow) | Legacy (DLT) |
|---|---|
| `AUTO CDC INTO target FROM source KEYS (id) SEQUENCE BY ts STORED AS SCD TYPE 2` | `APPLY CHANGES INTO target FROM source KEYS (id) SEQUENCE BY ts APPLY AS DELETE WHEN op = 'D' STORED AS SCD TYPE 2` |

### APPLY CHANGES INTO — Full Syntax

```sql
APPLY CHANGES INTO
  LIVE.silver_customers         -- target table (must be STREAMING TABLE)
FROM
  STREAM(LIVE.bronze_customers) -- CDC source stream
KEYS
  (customer_id)                 -- natural key(s) for matching rows
SEQUENCE BY
  updated_at                    -- ordering column (determines "latest" value)
IGNORE NULL UPDATES             -- optional: skip rows where all tracked cols are NULL
APPLY AS DELETE WHEN
  operation_type = 'DELETE'     -- expr that marks a delete event
APPLY AS TRUNCATE WHEN
  operation_type = 'TRUNCATE'   -- optional truncate support
STORED AS
  SCD TYPE 2;                   -- SCD1 (overwrite) or SCD2 (history)
```

### Key Clauses

| Clause | Required | Description |
|---|---|---|
| `INTO target` | [OK] | Target STREAMING TABLE |
| `FROM STREAM(source)` | [OK] | Input CDC stream |
| `KEYS (col,...)` | [OK] | Natural key — used to match target rows |
| `SEQUENCE BY col` | [OK] | Ordering (timestamp / version) for deduplication |
| `APPLY AS DELETE WHEN` | No | Identify delete events |
| `STORED AS SCD TYPE 1/2` | No | Type 1 = overwrite (default), Type 2 = history |

> **Pro Tip:** `APPLY CHANGES INTO` requires the target to be a `STREAMING TABLE`, not a `MATERIALIZED VIEW`. Missing `KEYS` or `SEQUENCE BY` causes a pipeline validation error.

---

## Summary

Key concepts and key concepts covered in this module.

---

| Topic | Key Concept | Key Terms |
|---|---|---|
| **Medallion** | Bronze → Silver → Gold | Raw, Validated, Business-ready |
| **SCD Type 1** | Overwrite, no history | `MERGE INTO ... UPDATE SET *` |
| **SCD Type 2** | Track history | `__START_AT`, `__END_AT`, `AUTO CDC` |
| **STREAMING TABLE** | Append-only, incremental | `CREATE STREAMING TABLE`, `STREAM()` |
| **MATERIALIZED VIEW** | Full recomputation | `CREATE MATERIALIZED VIEW` |
| **FLOW** | Separate source from table | `INSERT INTO ONCE` (backfill) |
| **Expectations** | Data quality constraints | `DROP ROW`, `FAIL UPDATE`, warn-only |
| **Quarantine** | Route invalid records separately | `is_quarantined`, `TEMPORARY`, `expect_all` |
| **Sinks** | Write to external targets | ForEachBatch, Kafka, Event Hubs |



---

### Resources

- [Databricks Lakeflow Docs](https://docs.databricks.com/en/delta-live-tables/index.html)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)
- [SCD with Lakeflow](https://docs.databricks.com/en/delta-live-tables/cdc.html)

---

← [02 — Batch & Stream Ingestion](02_batch_stream_ingestion_demo.ipynb) | **[ README](../../../README.md)** | [04 — Orchestration](04_orchestration_demo.ipynb) →
